# Unit 8 — Missing Data — Smart Handling, Not Mechanical 🟢 FLEXIBLE
**Phase 2 | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Financial data is **never clean**. But the fix is not always the same.

> Forward-filling a price? Reasonable.
> Forward-filling volume? **Dangerous**.
> Forward-filling a bankruptcy flag? Catastrophic.

The skill here is **judgment** — knowing when each treatment is appropriate.
The code is simple. The decision-making is what matters.


---
## 1️⃣ Detecting Missing Data

```python
# Core tools
df.isna()           # Boolean mask — True where NaN
df.isnull()         # Same as isna() — legacy alias
df.notna()          # Opposite
df.isna().sum()     # Count NaN per column
df.isna().mean()    # Fraction missing per column (%)
```


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

# Pull some data — yfinance can have gaps
tickers = ['SPY', 'AAPL', 'TSLA', 'BRK-B']
prices = yf.download(tickers, start='2018-01-01', auto_adjust=True)['Close']

print("Missing data by ticker:")
print(prices.isna().sum())
print(f"\n% missing:")
print((prices.isna().mean() * 100).round(2))


In [ ]:
# Detect gaps — consecutive missing days
# A gap of 1-3 days = likely holiday
# A gap of 30+ days = data problem or delisting

def analyze_gaps(series):
    # Find missing date gaps in a price series
    missing = series[series.isna()]
    if len(missing) == 0:
        return "No missing data"
    
    # Find consecutive groups
    gaps = []
    in_gap = False
    gap_start = None
    
    for date, val in series.items():
        if pd.isna(val):
            if not in_gap:
                gap_start = date
                in_gap = True
        else:
            if in_gap:
                gaps.append((gap_start, date, (date - gap_start).days))
                in_gap = False
    
    return pd.DataFrame(gaps, columns=['Gap_Start', 'Gap_End', 'Days'])

for ticker in tickers:
    result = analyze_gaps(prices[ticker])
    print(f"\n{ticker}:")
    print(result if isinstance(result, pd.DataFrame) else result)


---
## 2️⃣ The Decision Tree — What Treatment to Use

| Data Type | Treatment | Reasoning |
|-----------|-----------|-----------|
| Prices | `.ffill()` | Last known price is best estimate; markets close on holidays |
| Log Returns | Drop (`.dropna()`) | Can't fabricate a return — it was just zero volume |
| Volume | Leave as NaN or 0 | Never forward-fill — volume on a holiday was genuinely 0 |
| Fundamentals (P/E, EPS) | `.ffill()` | Q3 EPS applies until Q4 is reported |
| Alternative data | Case by case | Depends on the signal — document your assumption |

> ⚠️ **Never forward-fill volume.** If you do, you'll think a stock had average volume on a holiday when it had zero. This distorts volume-based signals like VWAP, volume surges, and liquidity filters.


In [ ]:
# Demonstrate the three main treatments

# Create a sample price series with deliberate gaps
dates = pd.date_range('2023-01-01', periods=10, freq='B')
prices_sample = pd.Series(
    [100, np.nan, np.nan, 103, 104, np.nan, 106, np.nan, 108, 109],
    index=dates,
    name='Price'
)
volume_sample = pd.Series(
    [1e6, np.nan, np.nan, 1.2e6, 0.9e6, np.nan, 1.1e6, np.nan, 1.3e6, 0.8e6],
    index=dates,
    name='Volume'
)

print("Original:")
print(pd.DataFrame({'Price': prices_sample, 'Volume': volume_sample}))


In [ ]:
# Forward fill — appropriate for prices
prices_ffill = prices_sample.ffill()

# Backward fill — useful for backfilling known future values (rare use case)
prices_bfill = prices_sample.bfill()

# Interpolation — linear between known points (use carefully!)
prices_interp = prices_sample.interpolate(method='linear')

# Drop — appropriate for returns
prices_dropped = prices_sample.dropna()

print("Forward filled (prices — ✅ appropriate):")
print(prices_ffill)

print("\nVolume — NEVER forward fill:")
print("❌ Don't do: volume_sample.ffill()")
print("✅ Do: leave as NaN or fill with 0")
print(volume_sample.fillna(0))  # Volume on holiday = 0


---
## 3️⃣ Missing Data Patterns — MCAR, MAR, MNAR

Understanding *why* data is missing matters for choosing treatment.

| Pattern | Full Name | Meaning | Example |
|---------|-----------|---------|---------|
| MCAR | Missing Completely At Random | No systematic pattern | Random API failures |
| MAR | Missing At Random | Missingness related to other observed data | Small-cap data missing when volume < threshold |
| MNAR | Missing Not At Random | Missingness related to the missing value itself | Stock stops reporting because it's in trouble |

> 💡 MNAR is the dangerous one. If a company stops filing financials because it's going bankrupt, forward-filling those financials introduces a silent error AND survivorship bias simultaneously.


In [ ]:
# Detect systematic missingness patterns

# Is data missing more on certain days of the week?
spy_full = yf.download('SPY', start='2015-01-01', auto_adjust=True)['Close'].squeeze()

# Check for any missing business days
biz_days = pd.date_range(spy_full.index.min(), spy_full.index.max(), freq='B')
missing_bdays = biz_days.difference(spy_full.index)

print(f"Business days in range: {len(biz_days)}")
print(f"Missing business days: {len(missing_bdays)}")

if len(missing_bdays) > 0:
    # Check which day of week is most often missing
    missing_df = pd.DataFrame({'Date': missing_bdays})
    missing_df['DayOfWeek'] = missing_df['Date'].dt.day_name()
    print("\nMissing by day of week:")
    print(missing_df['DayOfWeek'].value_counts())
    print("\n→ Mondays/Fridays? Likely US public holidays.")


In [ ]:
# Build a simple data quality report

def data_quality_report(df):
    # Generate a missing data summary for a multi-ticker DataFrame
    report = pd.DataFrame({
        'Total_Rows': len(df),
        'Missing': df.isna().sum(),
        'Pct_Missing': (df.isna().mean() * 100).round(2),
        'Longest_Gap': df.apply(lambda col: col.isna().astype(int)
                                 .groupby((~col.isna()).cumsum()).sum().max()),
        'First_Valid': df.apply(lambda col: col.first_valid_index()),
        'Last_Valid': df.apply(lambda col: col.last_valid_index()),
    })
    return report

prices_multi = yf.download(tickers, start='2018-01-01', auto_adjust=True)['Close']
report = data_quality_report(prices_multi)
print("Data Quality Report:")
print(report)


---
## ✅ Self-Check Questions

1. When is forward-filling appropriate vs dangerous?
   > *Appropriate: prices, fundamentals (where last known value is a valid estimate). Dangerous: volume, returns — you'd be fabricating observations that didn't exist.*

2. How do you detect if missingness is systematic (e.g., always missing on Mondays)?
   > *Check `missing_bdays.day_name().value_counts()` — any day significantly over-represented signals a pattern.*

3. What's the difference between `.fillna(method='ffill')` and `.ffill()`?
   > *They're equivalent — `.ffill()` is the modern shorthand. The `method=` syntax is deprecated in Pandas 2.2+.*

4. Why should you never forward-fill trading volume?
   > *Volume on a holiday was genuinely 0. Forward-filling implies the previous day's volume repeated — this distorts VWAP, volume-surge signals, and liquidity calculations.*

---
## 🧪 Optional Tasks

- Build the `data_quality_report()` function above for a universe of 10 tickers.
- Find the longest consecutive gap in any ticker's data. What happened on those dates?
- Deliberately introduce 20% random NaN values into a price series. Apply ffill. Check how far forward the fill propagates. Does it still make sense after 5+ days?
- Find a ticker that has data gaps that correlate with known events (halted trading, corporate events).
